In [33]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor   
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

import warnings
warnings.filterwarnings(action='ignore')

In [34]:
df=pd.read_csv('data/Property Prices in Tunisia.csv')
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12748 entries, 0 to 12747
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   category        12748 non-null  object 
 1   room_count      12748 non-null  float64
 2   bathroom_count  12748 non-null  float64
 3   size            12748 non-null  float64
 4   type            12748 non-null  object 
 5   price           12748 non-null  float64
 6   city            12748 non-null  object 
 7   region          12748 non-null  object 
 8   log_price       12748 non-null  float64
dtypes: float64(5), object(4)
memory usage: 896.5+ KB


,room_count,bathroom_count,size,price,log_price
count,12748.000000,12748.000000,12748.000000,1.274800e+04,12748.000000
mean,1.759649,0.759884,130.896219,1.601575e+07,4.374245
std,2.171468,1.264812,184.074990,1.016644e+09,1.389788
min,-1.000000,-1.000000,-1.000000,1.000000e+01,1.000000
25%,-1.000000,-1.000000,-1.000000,8.500000e+02,2.929419
50%,2.000000,1.000000,95.000000,8.975000e+04,4.953033
75%,3.000000,1.000000,150.000000,2.600000e+05,5.414973
max,20.000000,10.000000,2000.000000,1.000000e+11,11.000000


preprocessing

In [35]:
def preprocess_data(df):
    df = df.copy()
    
    # Encode missing values properly
    df = df.replace(-1, np.nan)
    
    # Fill missing values with column medians
    for column in ['room_count', 'bathroom_count', 'size']:
        df[column] = df[column].fillna(df[column].median())
    
    # Binary encoding
    df['type'] = df['type'].replace({'À Louer': 0, 'À Vendre': 1})
    
    # One-hot encoding
    for column in ['category', 'city', 'region']:
        dummies = pd.get_dummies(df[column], prefix=column)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    
    # Drop log_price column
    df = df.drop('log_price', axis=1)
    
    # Split df into X and y
    y = df['price']
    X = df.drop('price', axis=1)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)
    
    # Scale X
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)
    
    return X_train, X_test, y_train, y_test

In [36]:
X_train, X_test, y_train, y_test=preprocess_data(df)
X_train.head()


,room_count,bathroom_count,size,type,category_Appartements,category_Bureaux et Plateaux,category_Colocations,category_Locations de vacances,"category_Magasins, Commerces et Locaux industriels",category_Maisons et Villas,...,region_Tozeur,region_Tunis,region_Téboulba,region_Téboursouk,region_Utique,region_Zaghouan,region_Zaouit-Ksibat Thrayett,region_Zarzis,region_Zarzouna,region_Zéramdine
83,-0.594847,-0.415569,-0.526100,-1.240965,-0.762766,-0.195968,13.321411,-0.15142,-0.233407,-0.575323,...,-0.055091,-0.113253,-0.018339,-0.014973,-0.02594,-0.086323,-0.066256,-0.023678,-0.021177,-0.010587
1917,-0.594847,1.004080,-0.377567,-1.240965,-0.762766,-0.195968,-0.075067,-0.15142,-0.233407,1.738154,...,-0.055091,-0.113253,-0.018339,-0.014973,-0.02594,-0.086323,-0.066256,-0.023678,-0.021177,-0.010587
7857,0.122329,-0.415569,-0.139914,-1.240965,-0.762766,-0.195968,-0.075067,-0.15142,-0.233407,1.738154,...,-0.055091,-0.113253,-0.018339,-0.014973,-0.02594,-0.086323,-0.066256,-0.023678,-0.021177,-0.010587
5044,-0.594847,-0.415569,-0.377567,-1.240965,-0.762766,5.102881,-0.075067,-0.15142,-0.233407,-0.575323,...,-0.055091,-0.113253,-0.018339,-0.014973,-0.02594,-0.086323,-0.066256,-0.023678,-0.021177,-0.010587
2726,0.122329,-0.415569,-0.258740,-1.240965,1.311018,-0.195968,-0.075067,-0.15142,-0.233407,-0.575323,...,-0.055091,-0.113253,-0.018339,-0.014973,-0.02594,-0.086323,-0.066256,-0.023678,-0.021177,-0.010587


In [37]:
y_train.head()

83      130.0
1917    550.0
7857    600.0
5044    700.0
2726    600.0
Name: price, dtype: float64

training with different models

In [38]:
models = {
    "                     Linear Regression": LinearRegression(),
    " Linear Regression (L2 Regularization)": Ridge(),
    " Linear Regression (L1 Regularization)": Lasso(),
    "                   K-Nearest Neighbors": KNeighborsRegressor(),
    "                        Neural Network": MLPRegressor(),
    "                         Decision Tree": DecisionTreeRegressor(),
    "                         Random Forest": RandomForestRegressor(),
    "                     Gradient Boosting": GradientBoostingRegressor()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")

                     Linear Regression trained.
 Linear Regression (L2 Regularization) trained.
 Linear Regression (L1 Regularization) trained.
                   K-Nearest Neighbors trained.
                        Neural Network trained.
                         Decision Tree trained.
                         Random Forest trained.
                     Gradient Boosting trained.


In [39]:
for name, model in models.items():
    r2 = model.score(X_test, y_test)
    print(name + " R^2: {:.4f}".format(r2))

                     Linear Regression R^2: -0.0020
 Linear Regression (L2 Regularization) R^2: -0.0020
 Linear Regression (L1 Regularization) R^2: -0.0018
                   K-Nearest Neighbors R^2: -0.0308
                        Neural Network R^2: -0.0003
                         Decision Tree R^2: -0.0029
                         Random Forest R^2: -0.0036
                     Gradient Boosting R^2: -0.0015
